# Capstone: Anomaly Detection with MLflow Experiment Tracking

**Dataset:** [Kaggle Fraud Detection](https://www.kaggle.com/datasets/kartik2112/fraud-detection)

## Before you run this notebook

1. Install dependencies:
   ```bash
   pip install -r requirements.txt
   ```
2. Download `creditcard.csv` from Kaggle and place it in `capstone/data/creditcard.csv`
3. Start MLflow UI in a **separate terminal** (must be running on port 5000):
   ```bash
   cd capstone
   mlflow ui --port 5000
   ```
4. Open http://localhost:5000 to view experiments while this notebook runs.

## 1. Imports and Configuration

In [ ]:
import warnings
from pathlib import Path

import mlflow
import mlflow.sklearn
import mlflow.xgboost
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from mlflow.tracking import MlflowClient
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
EXPERIMENT_NAME = "anomaly-detection-fraud"
REGISTERED_MODEL_NAME = "anomaly-detector-xgb-smote"
PRODUCTION_MODEL_NAME = "anomaly-detection-prod"

# Paths relative to capstone/ folder
BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

DATA_PATH = BASE_DIR / "data" / "creditcard.csv"
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Data path: {DATA_PATH}")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT_NAME}")

## 2. Load Dataset and Inspect Class Distribution

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Download creditcard.csv from Kaggle and place it in capstone/data/"
    )

df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
TARGET_COL = "Class"

class_counts = df[TARGET_COL].value_counts()
class_ratio = df[TARGET_COL].value_counts(normalize=True)

print("Class counts:")
print(class_counts)
print("\nClass proportions:")
print(class_ratio)

majority_pct = class_ratio.max() * 100
minority_pct = class_ratio.min() * 100
print(f"\nImbalance ratio: {majority_pct:.2f}% : {minority_pct:.2f}%")
print(f"(Assignment requires at least 80:20 — this dataset is ~{majority_pct:.0f}:{minority_pct:.0f})")

## 3. Stratified Train-Test Split (70% / 30%)

The **test set stays untouched** until final evaluation in each experiment.

In [ ]:
X = df.drop(TARGET_COL, axis=1)
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")
print("\nTrain class distribution:")
print(y_train.value_counts(normalize=True))
print("\nTest class distribution:")
print(y_test.value_counts(normalize=True))

## 4. SMOTE on Training Set Only

In [ ]:
print("Before SMOTE (training set only):")
print(y_train.value_counts())

smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE (training set only):")
print(pd.Series(y_train_smote).value_counts())

print(f"\nTraining samples before SMOTE: {len(X_train)}")
print(f"Training samples after SMOTE:  {len(X_train_smote)}")

## 5. Metric Helper Function

In [ ]:
def get_metrics(y_true, y_pred):
    """Extract the four required metrics from classification_report."""
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        "accuracy": report["accuracy"],
        "recall_class_0": report["0"]["recall"],
        "recall_class_1": report["1"]["recall"],
        "f1_score_macro": report["macro avg"]["f1-score"],
    }


def log_metrics_to_mlflow(metrics: dict) -> None:
    for name, value in metrics.items():
        mlflow.log_metric(name, value)

## 6. Experiment 1 — Logistic Regression (Baseline)

In [ ]:
with mlflow.start_run(run_name="Logistic Regression"):
    lr_params = {"C": 1, "solver": "liblinear", "random_state": RANDOM_STATE}

    model = LogisticRegression(**lr_params)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    metrics = get_metrics(y_test, preds)

    mlflow.log_param("C", lr_params["C"])
    mlflow.log_param("solver", lr_params["solver"])
    mlflow.log_param("used_smote", False)
    log_metrics_to_mlflow(metrics)
    mlflow.sklearn.log_model(model, artifact_path="model")

    print("Logistic Regression metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

## 7. Experiment 2 — Random Forest

In [ ]:
with mlflow.start_run(run_name="Random Forest"):
    rf_params = {
        "n_estimators": 30,
        "max_depth": 3,
        "random_state": RANDOM_STATE,
    }

    model = RandomForestClassifier(**rf_params)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    metrics = get_metrics(y_test, preds)

    mlflow.log_param("n_estimators", rf_params["n_estimators"])
    mlflow.log_param("max_depth", rf_params["max_depth"])
    mlflow.log_param("used_smote", False)
    log_metrics_to_mlflow(metrics)
    mlflow.sklearn.log_model(model, artifact_path="model")

    print("Random Forest metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

## 8. Experiment 3 — XGBoost (Original Imbalanced Data)

In [ ]:
with mlflow.start_run(run_name="XGBClassifier"):
    xgb_params = {
        "eval_metric": "logloss",
        "random_state": RANDOM_STATE,
    }

    model = XGBClassifier(**xgb_params)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    metrics = get_metrics(y_test, preds)

    mlflow.log_param("eval_metric", xgb_params["eval_metric"])
    mlflow.log_param("used_smote", False)
    log_metrics_to_mlflow(metrics)
    mlflow.xgboost.log_model(model, artifact_path="model")

    print("XGBClassifier metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

## 9. Experiment 4 — XGBoost with SMOTE

In [ ]:
with mlflow.start_run(run_name="XGBClassifier With SMOTE"):
    xgb_params = {
        "eval_metric": "logloss",
        "random_state": RANDOM_STATE,
    }

    model = XGBClassifier(**xgb_params)
    model.fit(X_train_smote, y_train_smote)
    preds = model.predict(X_test)
    metrics = get_metrics(y_test, preds)

    mlflow.log_param("eval_metric", xgb_params["eval_metric"])
    mlflow.log_param("used_smote", True)
    log_metrics_to_mlflow(metrics)
    mlflow.xgboost.log_model(model, artifact_path="model")

    print("XGBClassifier With SMOTE metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

## 10. Compare All Runs

In [ ]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.recall_class_1 DESC", "metrics.f1_score_macro DESC"],
)

display_cols = [
    "run_id",
    "tags.mlflow.runName",
    "metrics.accuracy",
    "metrics.recall_class_0",
    "metrics.recall_class_1",
    "metrics.f1_score_macro",
]
available_cols = [c for c in display_cols if c in runs_df.columns]
comparison = runs_df[available_cols].copy()
comparison

## 11. Model Registry — Register Best Model

Best model is selected by **recall_class_1** first, then **f1_score_macro**.

In [ ]:
best_run = runs_df.iloc[0]
best_run_id = best_run["run_id"]
best_run_name = best_run.get("tags.mlflow.runName", "unknown")

print(f"Best run: {best_run_name}")
print(f"Run ID: {best_run_id}")
print(f"recall_class_1: {best_run['metrics.recall_class_1']:.4f}")
print(f"f1_score_macro: {best_run['metrics.f1_score_macro']:.4f}")

model_uri = f"runs:/{best_run_id}/model"

In [ ]:
client = MlflowClient()

# Register best model
registered = mlflow.register_model(model_uri, REGISTERED_MODEL_NAME)
challenger_version = registered.version
print(f"Registered {REGISTERED_MODEL_NAME} version {challenger_version}")

# Assign @challenger alias
client.set_registered_model_alias(
    REGISTERED_MODEL_NAME,
    "challenger",
    challenger_version,
)
print(f"Alias @challenger assigned to version {challenger_version}")

## 12. Copy to Production Registry and Assign @champion

In [ ]:
copied = client.copy_model_version(
    src_model_uri=f"models:/{REGISTERED_MODEL_NAME}/{challenger_version}",
    dst_name=PRODUCTION_MODEL_NAME,
)
prod_version = copied.version
print(f"Copied to {PRODUCTION_MODEL_NAME} version {prod_version}")

client.set_registered_model_alias(
    PRODUCTION_MODEL_NAME,
    "champion",
    prod_version,
)
print(f"Alias @champion assigned to {PRODUCTION_MODEL_NAME} version {prod_version}")

## 13. Load Production Model and Run Inference on Test Set

In [ ]:
prod_model_uri = f"models:/{PRODUCTION_MODEL_NAME}@champion"
prod_model = mlflow.pyfunc.load_model(prod_model_uri)

final_preds = prod_model.predict(X_test)
final_metrics = get_metrics(y_test, final_preds)

print(f"Production model loaded from: {prod_model_uri}")
print("\nFinal test-set metrics (production @champion):")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.4f}")

print("\nClassification report:")
print(classification_report(y_test, final_preds, zero_division=0))

## 14. Summary

| Step | Status |
|------|--------|
| 4 model experiments logged | Done |
| Metrics: accuracy, recall_class_0, recall_class_1, f1_score_macro | Done |
| Best model registered as `anomaly-detector-xgb-smote` | Done |
| `@challenger` alias assigned | Done |
| Copied to `anomaly-detection-prod` | Done |
| `@champion` alias assigned | Done |
| Production inference on test set | Done |

**Next steps for submission:**
1. Capture 5 MLflow UI screenshots → save in `capstone/screenshots/`
2. Write experiment report PDF → `capstone/reports/`
3. Record screencast walkthrough